In [0]:
%run ../00_setup/01_config

In [0]:
%run ./01_bronze_ingestion_framework

In [0]:
import uuid

# Widget for day_number, so you don't edit code between days
dbutils.widgets.text("day_number", "1")
day_number = int(dbutils.widgets.get("day_number"))

In [0]:
# One batch_id shared across every table in this run
batch_id = str(uuid.uuid4())
print(f"Starting Bronze run — day_number={day_number}, batch_id={batch_id}")

In [0]:
# Ingest tables in dependency order
table_order = [
    "merchant_reference",
    "market_rates",
    "customer_master",
    "account_master",
    "loan_master",
    "product_holdings",
    "transaction_fact",
    "card_transaction_fact",
    "account_balance_snapshot",
    "device_events",
    "digital_activity",
    "support_tickets",
]

for table in table_order:
    ingest_table(table, day_number, batch_id=batch_id)

In [0]:
# Summary of this run, pulled straight from the audit log
summary_df = spark.sql(f"""
    SELECT table_name, status, source_record_count, target_record_count, error_message
    FROM {DQ_CATALOG}.{DQ_SCHEMA}.pipeline_audit_log
    WHERE day_number = {day_number}
      AND run_id IN (
        SELECT run_id FROM {DQ_CATALOG}.{DQ_SCHEMA}.pipeline_audit_log
        WHERE day_number = {day_number}
      )
    ORDER BY started_at
""")

print(f"--- Bronze run summary (day {day_number}) ---")
display(summary_df)